# Moodify: does audio help the lyrics model?

Tests whether adding the song's **audio** to the live **BERT lyrics model** predicts valence ("feels good")
and arousal ("energy") better, on the same 256 MERGE test songs as the encoder comparison.

**What it does**
1. Downloads MERGE's 30-second audio clips from Zenodo. It checks the record's licence first and stops unless it is CC BY-NC.
2. Matches each clip to a song in your `va_lyrics.csv` by title and artist, and uses the same seed-42 split (2,044 / 256 / 256).
3. Turns each clip into features with two pretrained music models, frozen (no fine-tuning):
   - **MERT-v1-95M**, a music model, read in 5-second slices;
   - **LAION-CLAP**, an audio-text model, read in 10-second slices.

   Each clip's features are the average over its slices, and optionally how much the slices vary
   (the "contrast" idea: a quiet-then-loud clip varies a lot, a steady one doesn't).
4. Trains a small ridge-regression head on the training songs for each feature set, and picks the best one on the validation songs.
5. **Fusion:** on the validation songs, learns how much to trust the lyrics prediction vs the audio prediction for each axis.
6. Scores lyrics only, audio only and lyrics + audio on the test songs that have audio. A paired bootstrap shows whether the gain is more than noise.

**Before running**
1. Your Drive folder `moodify` needs `va_lyrics.csv` (already there from the last notebook) and `va_model_state.bin`
   (the live BERT weights, the same file the 100k notebook uses).
2. Runtime › Change runtime type › **T4 GPU**, then Runtime › Run all.
3. Expect roughly 30–60 minutes, mostly the audio download and feature extraction. Features are cached in
   `MyDrive/moodify/audio_cache/`, so a re-run after an interruption skips the slow part.

Nothing here prints or saves lyrics or audio. Results (numbers, titles and artists only) go to
`MyDrive/moodify/colab_results/audio_*`.

MERGE: Louro et al. (2024), [Zenodo 13939205](https://doi.org/10.5281/zenodo.13939205), CC BY-NC 4.0.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/moodify"
ZENODO_RECORD = "13939205"
SEED = 42
MAX_LEN, STRIDE, MAX_WINDOWS, BATCH = 160, 120, 8, 32   # live BERT, whole-song read
MERT_SLICE, MERT_HOP = 5.0, 2.5     # seconds
CLAP_SLICE, CLAP_HOP = 10.0, 5.0
ALPHAS = [0.1, 1, 10, 100, 1e3, 1e4, 1e5]

In [ ]:
# MERT ships its own model code, written for 2024-era transformers; the newest release fails to load it
# ("'NoneType' object is not iterable"), so pin a version from then.
!pip -q install "transformers==4.44.2" nnAudio librosa
from google.colab import drive
drive.mount("/content/drive")
import os, re, io, json, glob, time, zipfile, unicodedata, requests
import numpy as np, pandas as pd, torch
# A GPU is much faster, but the notebook also runs on Colab's free CPU runtime (slower: allow 1-3 hours).
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dev.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 2)
    print("No GPU: running on CPU. Cached embeddings in Drive are reused, so a rerun picks up where it stopped.")
OUT_DIR = os.path.join(DRIVE_DIR, "colab_results"); os.makedirs(OUT_DIR, exist_ok=True)
CACHE = os.path.join(DRIVE_DIR, "audio_cache"); os.makedirs(CACHE, exist_ok=True)
for f in ["va_lyrics.csv", "va_model_state.bin"]:
    assert os.path.exists(os.path.join(DRIVE_DIR, f)), f"upload {f} to {DRIVE_DIR}"
print(torch.cuda.get_device_name(0) if dev.type == "cuda" else "CPU")

## 1. MERGE audio: licence check and download

In [ ]:
rec = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=60).json()
lic = (rec.get("metadata", {}).get("license") or {})
lic_id = (lic.get("id") if isinstance(lic, dict) else str(lic)) or ""
print("Record:", rec["metadata"]["title"])
print("Licence:", lic_id)
assert "by-nc" in lic_id.lower(), f"Stopping: the record's licence is {lic_id!r}, not CC BY-NC. Tell Claude."

files = [(f["key"], f["size"], f["links"]["self"]) for f in rec["files"]]
for k, s, _ in files:
    print(f"  {s / 1e6:9.1f} MB  {k}")
audio = [f for f in files if re.search(r"audio", f[0], re.I) and f[0].lower().endswith(".zip")]
# Prefer the complete audio set; fall back to whatever audio archive the record has.
pick = [f for f in audio if re.search(r"complete", f[0], re.I)] or [f for f in audio if not re.search(r"balanced", f[0], re.I)] or audio
assert pick, "No audio archive found in the record; tell Claude which file above holds the audio."
pick = pick[:1]
print("Using:", [k for k, _, _ in pick])

In [ ]:
AUDIO_DIR = "/content/merge_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)
for key, size, url in pick:
    dest = os.path.join("/content", key)
    if not os.path.exists(dest) or os.path.getsize(dest) != size:
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(1 << 22):
                    fh.write(chunk)
    with zipfile.ZipFile(dest) as z:
        z.extractall(AUDIO_DIR)
clips = {os.path.splitext(os.path.basename(p))[0]: p
         for p in glob.glob(f"{AUDIO_DIR}/**/*", recursive=True)
         if p.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a"))}
print(f"{len(clips)} audio clips, e.g. {list(clips)[:3]}")

## 2. Match clips to the lyrics songs

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\(.*?\)|\[.*?\]", " ", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

# MERGE ships metadata CSVs with a song ID column whose values name the audio files.
meta = None
for p in glob.glob(f"{AUDIO_DIR}/**/*.csv", recursive=True) + glob.glob(f"{AUDIO_DIR}/**/*.xlsx", recursive=True):
    try:
        m = pd.read_csv(p) if p.endswith(".csv") else pd.read_excel(p)
    except Exception:
        continue
    cols = {c.lower(): c for c in m.columns}
    if "artist" in cols and "title" in cols:
        idcol = next((c for c in m.columns if m[c].astype(str).isin(clips.keys()).mean() > 0.5), None)
        if idcol:
            meta = m.rename(columns={cols["artist"]: "artist", cols["title"]: "title", idcol: "clip_id"})
            print("metadata:", os.path.relpath(p, AUDIO_DIR), "| id column:", idcol, "|", len(meta), "rows")
            break
assert meta is not None, "Couldn't find MERGE's metadata file with artist, title and a clip ID column; tell Claude."
meta["key"] = meta["artist"].map(norm) + "|" + meta["title"].map(norm)
meta = meta.drop_duplicates("key")

from sklearn.model_selection import train_test_split
df = pd.read_csv(os.path.join(DRIVE_DIR, "va_lyrics.csv")).dropna(subset=["lyrics", "valence", "arousal"])
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["quadrant"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["quadrant"])
print(f"split: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test")

# Same 256 test songs as the published comparison (titles and numbers only).
pub = pd.read_csv("https://raw.githubusercontent.com/jason-d-garcia/moodify/main/data/merge_test_scores_wholesong.csv")
assert len(pub) == len(test_df) and np.allclose(pub["valence"], test_df["valence"], atol=1e-3), "test split differs from the published one"

clip_of = dict(zip(meta["key"], meta["clip_id"].astype(str)))
for d in (train_df, val_df, test_df):
    d["clip"] = (d["artist"].map(norm) + "|" + d["title"].map(norm)).map(clip_of)
    d["clip"] = d["clip"].where(d["clip"].isin(clips.keys()))
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {d['clip'].notna().sum()} of {len(d)} songs have audio")

## 3. Live BERT lyrics predictions (whole-song read)

In [ ]:
from torch import nn
from transformers import BertModel, BertTokenizer

class VARegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-cased")
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        pooled = self.bert(input_ids=input_ids, attention_mask=attention_mask)["pooler_output"]
        return torch.tanh(self.out(self.drop(pooled)))

tok = BertTokenizer.from_pretrained("bert-base-cased")
bert = VARegressor()
bert.load_state_dict(torch.load(os.path.join(DRIVE_DIR, "va_model_state.bin"), map_location="cpu"))
bert.to(dev).eval()

def bert_predict(texts):
    body = MAX_LEN - 2; ids_all, owner = [], []
    for k, t in enumerate(texts):
        ids = tok(str(t), add_special_tokens=False, truncation=False, verbose=False)["input_ids"]
        starts = list(range(0, max(len(ids) - body, 0) + 1, STRIDE))
        if starts[-1] + body < len(ids): starts.append(len(ids) - body)
        for s in starts[:MAX_WINDOWS]:
            c = [tok.cls_token_id] + ids[s:s + body] + [tok.sep_token_id]
            ids_all.append(c + [tok.pad_token_id] * (MAX_LEN - len(c))); owner.append(k)
    ids_all = torch.tensor(ids_all); mask = (ids_all != tok.pad_token_id).long(); out = []
    with torch.no_grad():
        for i in range(0, len(ids_all), BATCH):
            out.append(bert(ids_all[i:i + BATCH].to(dev), mask[i:i + BATCH].to(dev)).float().cpu())
    p = torch.cat(out).numpy(); owner = np.array(owner)
    sums = np.zeros((len(texts), 2)); np.add.at(sums, owner, p)
    return sums / np.bincount(owner, minlength=len(texts))[:, None]

bert_val = bert_predict(val_df["lyrics"].to_numpy())
bert_test = bert_predict(test_df["lyrics"].to_numpy())
gap = np.abs(bert_test - pub[["pred_valence", "pred_arousal"]].to_numpy()).max()
print(f"largest gap vs the published live predictions: {gap:.4f}")
assert gap < 0.02, "BERT predictions don't match the live model; check va_model_state.bin"
del bert; torch.cuda.empty_cache()

## 4. Audio features (cached in Drive)

In [ ]:
import librosa
from transformers import AutoModel, Wav2Vec2FeatureExtractor, ClapModel, ClapProcessor

need = sorted(set(pd.concat([train_df, val_df, test_df])["clip"].dropna()))
print(len(need), "clips to embed")

def slices(y, sr, length, hop):
    n, h = int(length * sr), int(hop * sr)
    if len(y) <= n: return [np.pad(y, (0, n - len(y)))]
    return [y[s:s + n] for s in range(0, len(y) - n + 1, h)]

def embed_all(tag, sr, length, hop, fn):
    path = os.path.join(CACHE, f"{tag}.npz")
    done = dict(np.load(path, allow_pickle=True)) if os.path.exists(path) else {}
    todo = [c for c in need if c not in done]
    t0 = time.time()
    for i, c in enumerate(todo):
        try:
            y, _ = librosa.load(clips[c], sr=sr, mono=True)
        except Exception as e:
            print("skip", c, e); continue
        done[c] = fn(slices(y, sr, length, hop))
        if (i + 1) % 200 == 0 or i + 1 == len(todo):
            np.savez(path, **done)
            print(f"  {tag}: {i + 1}/{len(todo)}  ({(time.time() - t0) / 60:.1f} min)")
    return done

# MERT: per slice, average each layer over time -> [layers, 768]; per clip keep mean and std over slices.
# MERT ships its own model code; if it won't load on this transformers version, carry on with CLAP alone.
try:
    mert_proc = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
    mert = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True).to(dev).eval()
    def mert_fn(sl):
        x = mert_proc(sl, sampling_rate=24000, return_tensors="pt", padding=True)
        with torch.no_grad():
            hs = mert(x["input_values"].to(dev), output_hidden_states=True).hidden_states
        per = torch.stack([h.mean(1) for h in hs], 1).float().cpu().numpy()   # [slices, layers, 768]
        return np.stack([per.mean(0), per.std(0)]).astype(np.float16)         # [2, layers, 768]
    MERT = embed_all("mert95m", 24000, MERT_SLICE, MERT_HOP, mert_fn)
    del mert; torch.cuda.empty_cache()
except Exception as e:
    import traceback; traceback.print_exc()
    print("MERT unavailable, continuing with CLAP only:", repr(e)[:300])
    MERT = {}

clap_proc = ClapProcessor.from_pretrained("laion/larger_clap_music")
clap = ClapModel.from_pretrained("laion/larger_clap_music").to(dev).eval()
def clap_fn(sl):
    try:
        x = clap_proc(audio=sl, sampling_rate=48000, return_tensors="pt")
    except (TypeError, ValueError):
        x = clap_proc(audios=sl, sampling_rate=48000, return_tensors="pt")   # older transformers
    with torch.no_grad():
        out = clap.get_audio_features(**{k: v.to(dev) for k, v in x.items()})
    # Older transformers return the embedding tensor; v5+ return a model output whose
    # pooler_output is the (normalised) 512-d audio embedding.
    e = (out if torch.is_tensor(out) else out.pooler_output).float().cpu().numpy()
    return np.stack([e.mean(0), e.std(0)]).astype(np.float16)              # [2, 512]
CLAP = embed_all("clap_music", 48000, CLAP_SLICE, CLAP_HOP, clap_fn)
del clap; torch.cuda.empty_cache()
print("done:", len(MERT), "MERT clips,", len(CLAP), "CLAP clips")

## 5. Audio heads, chosen on validation

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def has(d, store): return d["clip"].isin(store.keys()).to_numpy()

def features(d, store, pick):
    return np.stack([pick(store[c].astype(np.float32)) for c in d["clip"][has(d, store)]])

def corr(p, t): return np.corrcoef(p, t)[0, 1]

n_layers = next(iter(MERT.values())).shape[1] if MERT else 0
CANDIDATES = {"CLAP mean": (CLAP, lambda e: e[0]), "CLAP mean+contrast": (CLAP, lambda e: e.reshape(-1))}
for L in range(n_layers):
    CANDIDATES[f"MERT layer {L} mean"] = (MERT, lambda e, L=L: e[0, L])
    CANDIDATES[f"MERT layer {L} mean+contrast"] = (MERT, lambda e, L=L: e[:, L].reshape(-1))

Y = lambda d, m: d[["valence", "arousal"]].to_numpy()[m]
rows, fitted = [], {}
for name, (store, pick) in CANDIDATES.items():
    mtr, mva = has(train_df, store), has(val_df, store)
    sc = StandardScaler().fit(features(train_df, store, pick))
    Xtr, Xva = sc.transform(features(train_df, store, pick)), sc.transform(features(val_df, store, pick))
    best = None
    for a in ALPHAS:
        m = Ridge(alpha=a).fit(Xtr, Y(train_df, mtr))
        p = np.clip(m.predict(Xva), -1, 1)
        mse = float(((p - Y(val_df, mva)) ** 2).mean())
        if best is None or mse < best[0]: best = (mse, a, m, p)
    mse, a, m, p = best
    fitted[name] = (store, pick, sc, m)
    rows.append({"features": name, "alpha": a, "val mse": mse,
                 "val r valence": corr(p[:, 0], Y(val_df, mva)[:, 0]), "val r arousal": corr(p[:, 1], Y(val_df, mva)[:, 1])})
cand = pd.DataFrame(rows).set_index("features").sort_values("val mse")
display(cand.head(12).round(3))
BEST = cand.index[0]
print("Best audio features on validation:", BEST)

## 6. Fusion, then the test songs

In [ ]:
from sklearn.linear_model import LinearRegression

def audio_pred(d, name):
    store, pick, sc, m = fitted[name]
    out = np.full((len(d), 2), np.nan); mk = has(d, store)
    out[mk] = np.clip(m.predict(sc.transform(features(d, store, pick))), -1, 1)
    return out

a_val, a_test = audio_pred(val_df, BEST), audio_pred(test_df, BEST)
mv = ~np.isnan(a_val[:, 0]); mt = ~np.isnan(a_test[:, 0])
yv, yt = val_df[["valence", "arousal"]].to_numpy(), test_df[["valence", "arousal"]].to_numpy()

# Per axis: y ~ w_lyrics * BERT + w_audio * audio + b, fitted on the validation songs only.
fused_test = bert_test.copy(); weights = {}
for j, ax in enumerate(["valence", "arousal"]):
    lr = LinearRegression().fit(np.c_[bert_val[mv, j], a_val[mv, j]], yv[mv, j])
    weights[ax] = {"lyrics": float(lr.coef_[0]), "audio": float(lr.coef_[1]), "bias": float(lr.intercept_)}
    fused_test[mt, j] = np.clip(lr.predict(np.c_[bert_test[mt, j], a_test[mt, j]]), -1, 1)
print("fusion weights:", json.dumps(weights, indent=1))

preds = {"Lyrics only (live BERT)": bert_test[mt], f"Audio only ({BEST})": a_test[mt], "Lyrics + audio": fused_test[mt]}
truth = yt[mt]
def summary(p, t):
    out = {}
    for j, ax in enumerate(["valence", "arousal"]):
        out[f"{ax} r"] = corr(p[:, j], t[:, j]); out[f"{ax} MAE"] = np.abs(p[:, j] - t[:, j]).mean()
        out[f"{ax} R²"] = 1 - ((p[:, j] - t[:, j]) ** 2).sum() / ((t[:, j] - t[:, j].mean()) ** 2).sum()
    return out
print(f"Test songs with audio: {mt.sum()} of {len(test_df)}")
results = pd.DataFrame({k: summary(p, truth) for k, p in preds.items()}).T.round(3)
display(results)

rng = np.random.default_rng(SEED); idx = rng.integers(0, len(truth), size=(2000, len(truth)))
boot = {}
for j, ax in enumerate(["valence", "arousal"]):
    d = np.array([corr(fused_test[mt][s, j], truth[s, j]) - corr(bert_test[mt][s, j], truth[s, j]) for s in idx])
    boot[ax] = {"Δr": corr(fused_test[mt][:, j], truth[:, j]) - corr(bert_test[mt][:, j], truth[:, j]),
                "95% CI": f"[{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]"}
print("Lyrics + audio minus lyrics only (paired bootstrap, 2,000 resamples). An interval above 0 is a real gain:")
display(pd.DataFrame(boot).T)

In [ ]:
# Save results: numbers, titles and artists only.
results.to_csv(os.path.join(OUT_DIR, "audio_fusion_test_metrics.csv"))
cand.to_csv(os.path.join(OUT_DIR, "audio_feature_candidates_val.csv"))
t = test_df[mt][["title", "artist", "valence", "arousal"]].copy()
t[["lyrics_v", "lyrics_a"]] = bert_test[mt]; t[["audio_v", "audio_a"]] = a_test[mt]; t[["fused_v", "fused_a"]] = fused_test[mt]
t.round(4).to_csv(os.path.join(OUT_DIR, "audio_fusion_test_predictions.csv"), index=False)
json.dump({"best_audio_features": BEST, "fusion_weights": weights, "bootstrap": boot,
           "n_test_with_audio": int(mt.sum()), "n_train_with_audio": int(has(train_df, fitted[BEST][0]).sum())},
          open(os.path.join(OUT_DIR, "audio_fusion_report.json"), "w"), indent=1)
# The chosen audio head (scaler + ridge), for scoring the whole catalogue in colab_audio_catalog.ipynb.
import joblib
_, _, sc_best, m_best = fitted[BEST]
joblib.dump({"features": BEST, "scaler": sc_best, "ridge": m_best, "fusion_weights": weights},
            os.path.join(OUT_DIR, "audio_head.joblib"))
print("saved to", OUT_DIR)